# Data Quality

## 1. Carga de los Datos 

In [26]:
# Importar Librerías Iniciales 

import sys
import os
from pathlib import Path

In [27]:
# Obtener la ruta del proyecto (subir un nivel desde notebooks/)
try:
    # Si el notebook está en la carpeta notebooks/
    project_root = Path(os.path.dirname(os.path.abspath('__file__'))).parent
except:
    # Si falla, usar la ruta manual
    project_root = Path.cwd().parent

In [28]:
# Agregar la raíz del proyecto al path
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

print(f"Ruta del proyecto: {project_root}")
print(f"Python path actualizado")

Ruta del proyecto: /Users/hanjeannettezamora/Trabajo-Final---M-dulo-06---Retail-Dataset
Python path actualizado


In [29]:
# Importar Librerías de Análisis de Datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import re

In [30]:
# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [31]:
#Importar Módulos del Proyecto
from ingestion.Ingesta import ingest_data
from data_quality.quality import clean_data, validate_data, run_data_quality_pipeline

print("Módulos del proyecto importados correctamente")
print("\n" + "="*60)


Módulos del proyecto importados correctamente



## 2. Exploración Inicial


In [32]:
df_raw.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [33]:
df_raw.tail(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541899,581587,22726,ALARM CLOCK BAKELIKE GREEN,4,2011-12-09 12:50:00,3.75,12680.0,France
541900,581587,22730,ALARM CLOCK BAKELIKE IVORY,4,2011-12-09 12:50:00,3.75,12680.0,France
541901,581587,22367,CHILDRENS APRON SPACEBOY DESIGN,8,2011-12-09 12:50:00,1.95,12680.0,France
541902,581587,22629,SPACEBOY LUNCH BOX,12,2011-12-09 12:50:00,1.95,12680.0,France
541903,581587,23256,CHILDRENS CUTLERY SPACEBOY,4,2011-12-09 12:50:00,4.15,12680.0,France
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [34]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [35]:
df_raw.describe(include='all')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


In [36]:
# Calcular nulos
nulos = df_raw.isnull().sum()
nulos_pct = (nulos / len(df_raw) * 100).round(2)

# Crear tabla solo con columnas que tienen nulos
tabla_nulos = pd.DataFrame({
    'Columna': nulos.index,
    'Valores nulos': nulos.values,
    'Porcentaje (%)': nulos_pct.values
})
# Filtrar solo columnas con nulos > 0
tabla_nulos = tabla_nulos[tabla_nulos['Valores nulos'] > 0].sort_values('Valores nulos', ascending=False)

# Mostrar
print("="*60)
print("Columnas con valores nulos:")
print("="*60)
print(tabla_nulos.to_string(index=False))
print("="*60)
print(f"Total de columnas con nulos: {len(tabla_nulos)}")
print(f"Total de registros con al menos un nulo: {df_raw.isnull().any(axis=1).sum():,}")
print("="*60)

Columnas con valores nulos:
    Columna  Valores nulos  Porcentaje (%)
 CustomerID         135080           24.93
Description           1454            0.27
Total de columnas con nulos: 2
Total de registros con al menos un nulo: 135,080


In [37]:
#Identificaciôn de filas completamente vacías 
filas_vacias = df_raw.isna().all(axis=1).sum()
print(f"\n Filas completamente vacías: {filas_vacias:,}")


 Filas completamente vacías: 0


A partir de la visualización inicial de la composición del dataset se logra identificar un total de 8 columnas y 541909 registros. Las columnas del dataset se componen por datos de facturación de las ventas. En este sentido, se logra observar la presencia de valores nulos en el dataset, particularmente en la columna CostumerID y Description. En el caso particular de los nulos de la columna CostumerID, es importante denotar que  estos valores corresponden a un 24.93% del total de los registros de la columna. En esta misma linea, los nulos presentes en CostumerID representan una problemática  de cara al objetivo de este trabajo. Lo anterior debido a que sin la información de CostumerID, no se puede realizar una segmentación de los clientes. Se opta por la eliminación de estos registros en lugar de asignar una ID temporal o imputar un valor en aras de evitar sesgos y posteriores distorciones en el análisis realizado. En el caso de la columna description se optará por la eliminación de los registros nulos debido al porcentaje bajo que representa sin aportar información relevante al análisis ateniente al caso. 

En el caso de la columna Quantity, correspondiente a la cantidad de cada producto por transacción, muestra la presencia de valores negativos. Lo mismo ocurre con la columna Unit Price, lo cual resulta inconsistente con el funcionamiento de los procesos de venta. En este sentido se opta igualmente por la eliminación de los valores en aras de evitar la inflación de las ventas y por consiguiente un sesgo en el proceso de análisis. 

In [39]:
# Duplicados

duplicados = df_raw.duplicated().sum()
duplicados_pct = (duplicados / len(df_raw) * 100).round(2)

print(f" Registros duplicados:")
print(f"   Cantidad: {duplicados:,}")
print(f"   Porcentaje: {duplicados_pct}%")

 Registros duplicados:
   Cantidad: 5,268
   Porcentaje: 0.97%


Los valores duplicados representan menos del 1% de los regisrtos totales. Estos seran eliminados para evitar distorciones y sesgos

## 3. Limpieza de los Datos

In [42]:
# Hacer una copia para no modificar el original
df_clean = df_raw.copy()
registros_iniciales = len(df_clean)

print("\n" + "="*60)
print("Limpieza de datos iniciada...")
print("="*60)
print(f" Registros iniciales: {registros_iniciales:,}")
print("="*60)


Limpieza de datos iniciada...
 Registros iniciales: 541,909


In [45]:
# Eliminar filas con CostumerID nulo
before = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID'])
after = len(df_clean)
eliminados = before - after
print(f"\n CustomerID nulos:")
print(f"   Eliminados: {eliminados:,} registros")
print(f"   Registros restantes: {after:,}")


 CustomerID nulos:
   Eliminados: 0 registros
   Registros restantes: 406,829


In [46]:
# Eliminar filas con Description nulo
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Description'])
after = len(df_clean)
eliminados = before - after
print(f"\n Description nulos:")
print(f"   Eliminados: {eliminados:,} registros")
print(f"   Registros restantes: {after:,}")


 Description nulos:
   Eliminados: 0 registros
   Registros restantes: 406,829


In [48]:
# Eliminar filas con Quantity <= 0
before = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] > 0]
after = len(df_clean)
eliminados = before - after
print(f"\n Quantity <= 0:")
print(f"   Eliminados: {eliminados:,} registros")
print(f"   Registros restantes: {after:,}")



 Quantity <= 0:
   Eliminados: 8,905 registros
   Registros restantes: 397,924


In [49]:
# Eliminar filas con UnitaryPrice <= 0
before = len(df_clean)
df_clean = df_clean[df_clean['UnitPrice'] > 0]
after = len(df_clean)
eliminados = before - after
print(f"\n UnitPrice <= 0:")
print(f"   Eliminados: {eliminados:,} registros")
print(f"   Registros restantes: {after:,}")



 UnitPrice <= 0:
   Eliminados: 40 registros
   Registros restantes: 397,884


In [50]:
#Eliminar filas duplicadas
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
eliminados = before - after
print(f"\n 5. Duplicados exactos:")
print(f"   Eliminados: {eliminados:,} registros")
print(f"   Registros restantes: {after:,}")


 5. Duplicados exactos:
   Eliminados: 5,192 registros
   Registros restantes: 392,692


In [51]:
# Ajuste de tipos de datos
df_clean['CustomerID'] = df_clean['CustomerID'].astype('int64')
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"\nTipos de datos convertidos:")
print(f"   CustomerID: int64")
print(f"   InvoiceDate: datetime64")


Tipos de datos convertidos:
   CustomerID: int64
   InvoiceDate: datetime64


In [53]:

print("\n" + "="*60)
print("Datos limpios y validados")
print("="*60)


print("\nColumnas de importancia sin nulos:")
columnas_criticas = ['InvoiceNo', 'Quantity', 'UnitPrice', 'CustomerID']
for col in columnas_criticas:
    nulos = df_clean[col].isnull().sum()
    if nulos == 0:
        print(f"    {col}: {nulos} nulos")
    else:
        print(f"    {col}: {nulos} nulos")


print("\n Valores positivos:")
if (df_clean['Quantity'] > 0).all():
    print("    Quantity: todos los valores son positivos")
else:
    print("    Quantity: hay valores negativos")

if (df_clean['UnitPrice'] > 0).all():
    print("    UnitPrice: todos los valores son positivos")
else:
    print("    UnitPrice: hay valores negativos")

duplicados = df_clean.duplicated().sum()
if duplicados == 0:
    print(f"\n   Duplicados: {duplicados} registros")
else:
    print(f"\n   Duplicados: {duplicados} registros")

# 4.4 Verificar tipos de datos
print("\n Tipos de datos:")
print(df_clean.dtypes)



Datos limpios y validados

Columnas de importancia sin nulos:
    InvoiceNo: 0 nulos
    Quantity: 0 nulos
    UnitPrice: 0 nulos
    CustomerID: 0 nulos

 Valores positivos:
    Quantity: todos los valores son positivos
    UnitPrice: todos los valores son positivos

   Duplicados: 0 registros

 Tipos de datos:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object


In [54]:
#Guardar el DataFrame limpio 
# Crear carpeta si no existe
processed_path = Path("data/processed/")
processed_path.mkdir(parents=True, exist_ok=True)

# Guardar en CSV
file_path = processed_path / "online_retail_clean.csv"
df_clean.to_csv(file_path, index=False)

print(f"\n Datos limpios guardados en: {file_path}")
print(f"   Registros: {len(df_clean):,}")
print(f"   Tamaño: {file_path.stat().st_size / 1024 / 1024:.2f} MB")



 Datos limpios guardados en: data/processed/online_retail_clean.csv
   Registros: 392,692
   Tamaño: 33.07 MB
